# Tiền xử lý dữ liệu phụ đề (.srt)

Notebook này có nhiệm vụ đọc các file phụ đề `.srt`, loại bỏ những thành phần không cần thiết và chuyển đổi chúng thành các file văn bản sạch (`.txt`) để phục vụ cho các bước xử lý dữ liệu tiếp theo.

---


# Giới thiệu cơ bản về File `.srt`

`.srt` (SubRip Subtitle) là định dạng file phụ đề video phổ biến và đơn giản nhất hiện nay. Mặc dù được sử dụng cho video, nhưng về bản chất nó chỉ là một **file văn bản thuần túy (plain text)** mà bạn có thể mở và chỉnh sửa bằng:

- **Notepad** (Windows)
- **TextEdit** (MacOS)

---

# 1. Cấu trúc của một file `.srt`

Một file `.srt` được chia thành các **khối (block)** lặp đi lặp lại. Mỗi khối đại diện cho một câu phụ đề hiển thị trên màn hình và bao gồm 4 thành phần:

## Thành phần của một khối phụ đề

### 1. Số thứ tự (Sequence Number)
Đánh dấu vị trí của câu phụ đề.

Ví dụ:

```text
1
2
3
...
```

### 2. Mốc thời gian (Timecode)

Xác định thời điểm phụ đề xuất hiện và biến mất trên màn hình.

Định dạng:

```text
Giờ:Phút:Giây,Mili-giây --> Giờ:Phút:Giây,Mili-giây
```

Ví dụ:

```text
00:00:01,500 --> 00:00:03,200
```

### 3. Nội dung (Subtitle Text)

Lời thoại hoặc nội dung hiển thị, thường từ 1–2 dòng.

Ví dụ:

```text
Bữa tối sẵn sàng rồi, Sheldon!
```

### 4. Dòng trống (Blank Line)

Dùng để ngăn cách giữa hai khối phụ đề liên tiếp.

---

## Ví dụ thực tế

```text
1
00:00:01,500 --> 00:00:03,200
Bữa tối sẵn sàng rồi, Sheldon!

2
00:00:03,500 --> 00:00:06,100
Con không ăn đâu, con đang bận.
```

---

# 2. Tại sao phải xử lý file `.srt` trong Machine Learning?

Trong bài toán **Dự đoán từ tiếp theo (Next Word Prediction)** bằng **RNN (Recurrent Neural Network)**, mục tiêu của mô hình là học được quy luật của **ngôn ngữ tự nhiên**, tức là phần **lời thoại**.

Tuy nhiên, nếu đưa nguyên file `.srt` vào mô hình, máy tính sẽ học lẫn cả:

- Số thứ tự phụ đề (`1`, `2`, `3`, ...)
- Mốc thời gian (`00:00:01,500`)
- Các ký tự đặc biệt liên quan đến định dạng phụ đề

Những thông tin này **không mang ý nghĩa ngôn ngữ** và sẽ gây **nhiễu dữ liệu (noise)**.

## Ví dụ

### Dữ liệu gốc (.srt)

```text
1
00:00:01,500 --> 00:00:03,200
Bữa tối sẵn sàng rồi, Sheldon!

2
00:00:03,500 --> 00:00:06,100
Con không ăn đâu, con đang bận.
```

### Dữ liệu sau khi tiền xử lý (.txt)

```text
Bữa tối sẵn sàng rồi Sheldon
Con không ăn đâu con đang bận
```

---

# Kết luận

Bước **Tiền xử lý dữ liệu (Preprocessing)** là bắt buộc khi làm việc với file `.srt` trong Machine Learning.

Mục tiêu của bước này là:

- Loại bỏ số thứ tự phụ đề.
- Loại bỏ mốc thời gian.
- Loại bỏ ký tự không cần thiết.
- Chuẩn hóa văn bản.

Kết quả cuối cùng là một file văn bản sạch (`.txt`) chỉ chứa nội dung lời thoại để phục vụ quá trình huấn luyện mô hình ngôn ngữ như **RNN**, **LSTM** hoặc **Transformer**.

## 1. Thiết lập thư mục làm việc

Đầu tiên, notebook khai báo các thư mục đầu vào và đầu ra.

- **Thư mục đầu vào (`SUB_DIR`)**
  - Được đặt tên là **`Sub`**.
  - Chứa toàn bộ **22 file phụ đề gốc** của dataset.

- **Thư mục đầu ra (`OUTPUT_DIR`)**
  - Được đặt tên là **`Cleaned_Text`**.
  - Nếu thư mục chưa tồn tại, chương trình sẽ tự động tạo mới.
  - Đây là nơi lưu các file văn bản đã được làm sạch sau khi xử lý.



In [1]:
import os      # Làm việc với thư mục và file
import re      # Xử lý biểu thức chính quy
import glob    # Tìm nhiều file theo mẫu

# Thư mục chứa file phụ đề gốc
SUB_DIR = "Sub"

# Thư mục lưu dữ liệu sau khi làm sạch
OUTPUT_DIR = "Cleaned_Text"

# Tạo thư mục đầu ra nếu chưa tồn tại
os.makedirs(OUTPUT_DIR, exist_ok=True)

# In thông tin các thư mục đang sử dụng
print(f"Directories configured: Input={SUB_DIR}, Output={OUTPUT_DIR}")


Directories configured: Input=Sub, Output=Cleaned_Text


## 2. Các hàm cốt lõi để làm sạch dữ liệu

Cell này bao gồm hai hàm chính chịu trách nhiệm xử lý nội dung của các file phụ đề.

### 2.1. `clean_subtitle_line(line)` – Bộ lọc dữ liệu

Hàm này sử dụng thư viện **Regular Expression (`re`)** để loại bỏ các thành phần không cần thiết trong lời thoại.

Các bước xử lý gồm:

- **Xóa thẻ HTML**
  - Loại bỏ các thẻ định dạng như:
    - `<i>`
    - `</i>`
    - `<b>`
    - `</b>`

- **Xóa mô tả hành động hoặc biểu cảm**
  - Loại bỏ nội dung nằm trong:
    - `( ... )`
    - `[ ... ]`
  - Ví dụ:
    - `[door opens]`
    - `(laughing)`

- **Xóa ký hiệu âm nhạc**
  - Loại bỏ các ký hiệu như:
    - `♪`
    - `#`

- **Xóa tên người nói**
  - Nhận diện và loại bỏ tên nhân vật đứng ở đầu câu.
  - Ví dụ:
    - `MARY:`
    - `SHELDON:`
    - `- ADULT SHELDON:`

- **Xóa dấu gạch đầu dòng**
  - Loại bỏ dấu `-` xuất hiện ở đầu lời thoại.

- **Chuẩn hóa khoảng trắng**
  - Loại bỏ khoảng trắng thừa.
  - Ghép câu thành một chuỗi văn bản sạch và liền mạch.

---

### 2.2. `parse_srt_file(filepath)` – Hàm đọc file `.srt`

Hàm này chịu trách nhiệm đọc và trích xuất nội dung từ file phụ đề.

Quy trình xử lý:

1. Đọc toàn bộ file `.srt`.

2. Chia nội dung thành từng **khối phụ đề (subtitle block)** dựa trên các dòng trống.

3. Trong mỗi khối:
   - Tìm dòng chứa ký hiệu `-->`, đây là dòng biểu thị mốc thời gian.
   - Bỏ qua:
     - số thứ tự,
     - mốc thời gian.

4. Ghép toàn bộ các dòng lời thoại còn lại thành một câu hoàn chỉnh.

5. Chuyển câu thoại sang hàm `clean_subtitle_line()` để làm sạch dữ liệu.

6. Trả về danh sách các câu thoại đã được xử lý.


In [2]:
def clean_subtitle_line(line):
    # Xóa thẻ HTML (<i>, <b>, ...)
    line = re.sub(r'<[^>]+>', '', line)

    # Xóa nội dung trong () và []
    line = re.sub(r'\([^)]*\)', '', line)
    line = re.sub(r'\[[^\]]*\]', '', line)

    # Xóa ký hiệu nhạc và ký tự nhiễu
    line = re.sub(r'[♪#]', '', line)

    # Xóa tên nhân vật (MARY:, SHELDON:, ...)
    line = re.sub(r'^\s*(?:-\s*)?[A-Z][A-Z0-9\s\'\.\-]{0,25}:\s*', '', line)

    # Xóa dấu gạch đầu dòng hội thoại
    line = re.sub(r'^\s*-\s*', '', line)

    # Chuẩn hóa khoảng trắng
    line = re.sub(r'\s+', ' ', line).strip()

    # Xóa dấu : hoặc khoảng trắng dư ở đầu câu
    line = re.sub(r'^[:\s]+', '', line).strip()

    return line


def parse_srt_file(filepath):
    # Đọc nội dung file .srt
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()

    # Tách các block phụ đề
    blocks = re.split(r'\n\s*\n', content)
    cleaned_lines = []

    # Duyệt từng block phụ đề
    for block in blocks:
        lines = [line.strip() for line in block.split('\n') if line.strip()]

        # Bỏ qua block rỗng
        if not lines:
            continue

        # Tìm dòng chứa timestamp
        time_idx = -1
        for idx, line in enumerate(lines):
            if '-->' in line:
                time_idx = idx
                break

        # Lấy phần lời thoại sau timestamp
        if time_idx != -1:
            text_lines = lines[time_idx + 1:]
        else:
            # Nếu không có timestamp, bỏ dòng số thứ tự
            text_lines = [line for line in lines if not line.isdigit()]

        # Ghép nhiều dòng thoại thành một dòng
        block_text = " ".join(text_lines)

        # Làm sạch dữ liệu
        cleaned = clean_subtitle_line(block_text)

        # Lưu nếu còn nội dung
        if cleaned:
            cleaned_lines.append(cleaned)

    return cleaned_lines


## 3. Xử lý hàng loạt toàn bộ dataset

Sau khi xây dựng các hàm xử lý, notebook sẽ tự động làm sạch toàn bộ dataset.

Quy trình gồm:

1. Sử dụng thư viện **`glob`** để quét toàn bộ các file có phần mở rộng `.srt` trong thư mục **`Sub`**.

2. Duyệt qua từng file bằng vòng lặp `for`.

3. Trích xuất tên tập phim (ví dụ: `S01E01`).

4. Gọi hàm `parse_srt_file()` để:
   - đọc nội dung,
   - làm sạch lời thoại,
   - thu được danh sách văn bản đã xử lý.

5. Lưu kết quả dưới dạng file `.txt` vào thư mục **`Cleaned_Text`**.

Ví dụ:

```text
S01E01.srt
        ↓
Làm sạch dữ liệu
        ↓
Cleaned_Text/S01E01.txt
```

6. Sau khi xử lý xong từng tập, chương trình sẽ in ra thông báo bao gồm:
   - tên tập phim,
   - số lượng dòng thoại đã được trích xuất,
   - trạng thái hoàn thành.

---


In [ ]:
# Lấy danh sách tất cả file .srt trong thư mục đầu vào
srt_files = glob.glob(os.path.join(SUB_DIR, "*.srt"))

# Hiển thị số lượng file tìm thấy
print(f"Found {len(srt_files)} subtitle (.srt) files in '{SUB_DIR}'.")

# Duyệt từng file phụ đề
for filepath in sorted(srt_files):

    # Lấy tên file
    filename = os.path.basename(filepath)

    # Trích xuất mã tập phim (S01E01, S01E02, ...)
    match = re.search(r'S\d+E\d+', filename, re.IGNORECASE)

    if match:
        ep_name = match.group(0).upper()
    else:
        # Nếu không tìm thấy, dùng tên file gốc
        ep_name = os.path.splitext(filename)[0]

    # Đọc và làm sạch dữ liệu phụ đề
    cleaned_lines = parse_srt_file(filepath)

    # Tạo đường dẫn file đầu ra
    output_path = os.path.join(OUTPUT_DIR, f"{ep_name}.txt")

    # Ghi dữ liệu đã làm sạch vào file .txt
    with open(output_path, 'w', encoding='utf-8') as f:
        for line in cleaned_lines:
            f.write(line + "\n")

    # Hiển thị kết quả xử lý
    print(f"Processed: {filename} -> {ep_name}.txt ({len(cleaned_lines)} lines)")

# Thông báo hoàn thành
print("\nAll subtitle files successfully cleaned and exported to 'Cleaned_Text/'!")
